# 06 - Search Methods: Test Results

This notebook runs every search method once on the 120 test questions, with the settings that were fixed on the 30 dev questions in notebooks 03 to 05.

Then, the results are compared against the following two baselines:

- BM25 is the internal baseline, the standard keyword search that the other methods have to beat.
- The published results of Kobeissi and Langlais (2026) are the external baseline. They evaluate the same method families (BM25, dense search, hybrid fusion, reranking) on the same 150 FinanceBench questions.

The notebook produces the retrieval results table for the report and the retrieval files that the reading notebooks use.

This notebook runs on Google Colab with a T4 GPU.

In [1]:
!git clone https://github.com/dogamentese/financebench-rag.git
%cd financebench-rag
!pip install -q -U sentence-transformers qdrant-client
!pip install -q pymupdf bm25s
!python scripts/download_data.py

Cloning into 'financebench-rag'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 60 (delta 20), reused 48 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 680.04 KiB | 14.17 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/financebench-rag
[1/84] 3M_2018_10K
[2/84] 3M_2022_10K
[3/84] 3M_2023Q2_10Q
[4/84] ACTIVISIONBLIZZARD_2019_10K
[5/84] ADOBE_2015_10K
[6/84] ADOBE_2016_10K
[7/84] ADOBE_2017_10K
[8/84] ADOBE_2022_10K
[9/84] AES_2022_10K
[10/84] AMAZON_2017_10K
[11/84] AMAZON_2019_10K
[12/84] AMCOR_2020_10K
[13/84] AMCOR_2022_8K_dated-2022-07-01
[14/84] AMCOR_2023Q2_10Q
[15/84] AMCOR_2023Q4_EARNINGS
[16/84] AMCOR_2023_10K
[17/84] AMD_2015_10K
[18/84] AMD_2022_10K
[19/84] AMERICANEXPRESS_2022_10K
[20/84] AMERICANWATERWORKS_2020_10K
[21/84] AMERICANWATERWORKS_2021_10K
[22/84] AMERICANWATERWORKS_2022_10K
[23/84] BESTBUY_2017_10K
[24/84] BESTBUY_2019_10K
[25/8

In [2]:
import sys
sys.path.append("/content/financebench-rag/src")

## 1. Frozen settings

Every setting below was chosen on the dev questions in the previous notebooks:

| Setting | Value | Chosen in |
|---|---|---|
| Chunks | 250 words, 50 words overlap, prefix with company, document type and year | notebook 03 |
| BM25 | default parameters of `bm25s` | notebook 03 |
| Embedding model | Qwen3-Embedding-0.6B, with the query prompt for questions | notebook 04 |
| Fusion | RRF with k = 60 over the top 100 pages of BM25 and embeddings | notebook 05 |
| Reranker | bge-reranker-v2-m3 on the top 20 candidates, a page scored by its best chunk | notebook 05 |

Five methods are compared in this notebook:

- BM25 (notebook 03)
- embeddings (notebook 04)
- fused (notebook 05)
- fused + rerank (notebook 05)
- embeddings + rerank (new)

Embeddings + rerank reranks the top 20 of the embedding list instead of the fused list, because notebook 05 showed that fusion puts fewer evidence pages in the candidate list. It uses the same reranker settings as fused + rerank.

In [3]:
import json
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from financebench_rag.data import REPO, extract_pages, gold_pages, load_doc_info, load_pages, load_questions, load_split
from financebench_rag.embedding import EmbeddingSearch
from financebench_rag.evaluate import evaluate_retrieval
from financebench_rag.retrieval import BM25Search, build_chunks, rrf_fuse

questions = load_questions()
split = load_split()
dev = [q for q in questions if q["financebench_id"] in set(split["dev"])]
test = [q for q in questions if q["financebench_id"] in set(split["test"])]
doc_info = load_doc_info()
print(f"{len(dev)} dev questions, {len(test)} test questions")

extract_pages()
pages = load_pages()
chunks = build_chunks(pages, doc_info, size=250, overlap=50, prefix=True)

bm25 = BM25Search(chunks)
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", model_kwargs={"dtype": "float16"})
model.max_seq_length = 512
start = time.time()
embeddings = EmbeddingSearch(chunks, model)
print(f"both searches ready in {(time.time() - start) / 60:.1f} min")

30 dev questions, 120 test questions


DEBUG:bm25s:Building index from IDs objects


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

... (8 outputs left out)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/532 [00:00<?, ?it/s]

/content/financebench-rag/src/financebench_rag/embedding.py:18: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert("chunks", points)


both searches ready in 29.2 min


## 2. Run every method

All 150 questions are searched in one pass (120 test questions + 30 dev questions).

The 30 dev questions are included because the published baselines are reported on all 150 questions.

In [4]:
deep_bm25 = {q["financebench_id"]: bm25.search(q["question"], k=100, n_chunks=2000) for q in questions}
deep_emb = {q["financebench_id"]: embeddings.search(q["question"], k=100, n_chunks=2000) for q in questions}
fused = {fid: rrf_fuse([deep_bm25[fid], deep_emb[fid]]) for fid in deep_bm25}
print("BM25, embeddings and fused rankings are ready")

BM25, embeddings and fused rankings are ready


In [5]:
# The embedding model is no longer needed once the ranked lists are computed,
# so it is removed from the GPU to make room for the reranker.
import gc

import torch

del model
embeddings.model = None
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory now: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

GPU memory now: 0.0 GB


In [6]:
from financebench_rag.rerank import Reranker

chunks_by_page = defaultdict(list)
for chunk in chunks:
    chunks_by_page[(chunk["doc_name"], chunk["page_num"])].append(chunk)

reranker = Reranker()
start = time.time()
fused_rerank = {q["financebench_id"]: reranker.rerank(q["question"], fused[q["financebench_id"]][:20], chunks_by_page) for q in questions}
emb_rerank = {q["financebench_id"]: reranker.rerank(q["question"], deep_emb[q["financebench_id"]][:20], chunks_by_page) for q in questions}
print(f"both rerankings done in {(time.time() - start) / 60:.1f} min")

rankings = {
    "BM25": {fid: ranked[:10] for fid, ranked in deep_bm25.items()},
    "embeddings": {fid: ranked[:10] for fid, ranked in deep_emb.items()},
    "fused (RRF)": {fid: ranked[:10] for fid, ranked in fused.items()},
    "embeddings + rerank": emb_rerank,
    "fused + rerank": fused_rerank,
}

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

both rerankings done in 27.6 min


The rankings are saved and downloaded before any analysis since files on Colab disappear when the session ends.

In [7]:
slugs = {
    "BM25": "bm25",
    "embeddings": "embeddings",
    "fused (RRF)": "fused",
    "fused + rerank": "hybrid",
    "embeddings + rerank": "embeddings_rerank",
}
test_ids = {q["financebench_id"] for q in test}
dev_ids = {q["financebench_id"] for q in dev}

for name, slug in slugs.items():
    path = REPO / "results" / f"test_retrieval_{slug}.json"
    path.write_text(json.dumps({fid: ranked for fid, ranked in rankings[name].items() if fid in test_ids}) + "\n")
path = REPO / "results" / "dev_retrieval_embeddings_rerank.json"
path.write_text(json.dumps({fid: ranked for fid, ranked in rankings["embeddings + rerank"].items() if fid in dev_ids}) + "\n")

!zip -q -j retrieval_results.zip results/test_retrieval_*.json results/dev_retrieval_embeddings_rerank.json
from google.colab import files
files.download("retrieval_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Check against the dev notebooks

The dev scores of this run are compared with the files saved in notebooks 03 to 05. Embeddings + rerank has no saved file, since it is newly developed.

The method with the highest dev recall@5 is passed to the next phase of the project.

In [8]:
def load_saved(name):
    return {fid: [tuple(page) for page in ranked] for fid, ranked in json.load(open(REPO / "results" / name)).items()}


saved = {"BM25": "dev_retrieval_bm25.json", "embeddings": "dev_retrieval_embeddings.json", "fused + rerank": "dev_retrieval_hybrid.json"}
rows = []
for name, ranking in rankings.items():
    rows.append({"method": name, "source": "this run", **evaluate_retrieval(ranking, dev)})
    if name in saved:
        rows.append({"method": name, "source": "saved file", **evaluate_retrieval(load_saved(saved[name]), dev)})
display(pd.DataFrame(rows))

dev_recall = {name: evaluate_retrieval(ranking, dev)["recall@5"] for name, ranking in rankings.items()}
handoff = max(dev_recall, key=dev_recall.get)
print("method handed to the reading notebooks:", handoff)

,method,source,recall@1,recall@5,mrr@10,doc_recall@5
0,BM25,this run,0.100,0.133,0.111,0.600
1,BM25,saved file,0.100,0.133,0.111,0.600
2,embeddings,this run,0.233,0.500,0.351,1.000
3,embeddings,saved file,0.233,0.500,0.351,1.000
4,fused (RRF),this run,0.100,0.233,0.161,0.767
5,embeddings + rerank,this run,0.200,0.467,0.344,0.967
6,fused + rerank,this run,0.200,0.533,0.334,0.933
7,fused + rerank,saved file,0.200,0.533,0.334,0.933


method handed to the reading notebooks: fused + rerank


The scores of this run match the saved files, so the results reproduce. Fused + rerank has the highest dev recall@5 (0.533) and is passed to the next phase of the project.

## 4. Test results

The search methods are scored on the 120 test questions.

Recall@5 and `page_rec@5` come with a 95% bootstrap confidence interval, from 10,000 resamples of the test questions.

`page_rec@5` is the fraction of a question's evidence pages in the top 5. It is added so that the results can be compared with the published results of Kobeissi and Langlais (2026).

In [9]:
rng = np.random.default_rng(0)


def hits_at_5(ranking, qs):
    """One True or False per question: is an evidence page among the top 5?"""
    return np.array([any(page in gold_pages(q) for page in ranking[q["financebench_id"]][:5]) for q in qs])


def bootstrap_ci(values, n=10000):
    """95% confidence interval of the mean, from n resamples with replacement."""
    means = rng.choice(values, size=(n, len(values))).mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return f"[{low:.3f}, {high:.3f}]"


def page_fractions(ranking, qs, k=5):
    """Per question, the fraction of its evidence pages among the top k (PageRec@k of Kobeissi and Langlais)."""
    return np.array([len(gold_pages(q) & set(ranking[q["financebench_id"]][:k])) / len(gold_pages(q)) for q in qs])


pd.DataFrame([
    {
        "method": name,
        **evaluate_retrieval(ranking, test),
        "recall@5 95% CI": bootstrap_ci(hits_at_5(ranking, test)),
        "page_rec@5": round(page_fractions(ranking, test).mean(), 3),
        "page_rec@5 95% CI": bootstrap_ci(page_fractions(ranking, test)),
    }
    for name, ranking in rankings.items()
])

,method,recall@1,recall@5,mrr@10,doc_recall@5,recall@5 95% CI,page_rec@5,page_rec@5 95% CI
0,BM25,0.083,0.158,0.112,0.658,"[0.092, 0.225]",0.158,"[0.100, 0.225]"
1,embeddings,0.283,0.675,0.444,0.958,"[0.592, 0.758]",0.628,"[0.543, 0.707]"
2,fused (RRF),0.183,0.358,0.266,0.950,"[0.275, 0.442]",0.346,"[0.263, 0.433]"
3,embeddings + rerank,0.300,0.675,0.468,0.983,"[0.592, 0.758]",0.642,"[0.558, 0.721]"
4,fused + rerank,0.292,0.617,0.421,0.983,"[0.533, 0.700]",0.583,"[0.500, 0.667]"


The table below compares each method with embeddings question by question. It counts the questions that only one of the two finds in the top 5, and gives a confidence interval for the difference in recall@5. If the interval does not include 0, the difference is unlikely to be chance.

In [10]:
base = hits_at_5(rankings["embeddings"], test).astype(int)
rows = []
for name in ["BM25", "fused (RRF)", "embeddings + rerank", "fused + rerank"]:
    diff = hits_at_5(rankings[name], test).astype(int) - base
    rows.append({
        "method": name,
        "found only by this method": int((diff == 1).sum()),
        "found only by embeddings": int((diff == -1).sum()),
        "difference in recall@5": round(diff.mean(), 3),
        "95% CI of the difference": bootstrap_ci(diff),
    })
pd.DataFrame(rows)

,method,found only by this method,found only by embeddings,difference in recall@5,95% CI of the difference
0,BM25,1,63,-0.517,"[-0.608, -0.425]"
1,fused (RRF),5,43,-0.317,"[-0.417, -0.217]"
2,embeddings + rerank,17,17,0.000,"[-0.092, 0.100]"
3,fused + rerank,13,20,-0.058,"[-0.150, 0.033]"


## 5. Results by question type and document type

Recall@5 on the test questions, split by the official question type and by document type.

In [11]:
per_question = pd.DataFrame({name: hits_at_5(ranking, test) for name, ranking in rankings.items()})
per_question["question_type"] = [q["question_type"] for q in test]
per_question["doc_type"] = [doc_info[q["doc_name"]]["doc_type"] for q in test]

for column in ["question_type", "doc_type"]:
    table = per_question.groupby(column)[list(rankings)].mean().round(3)
    table.insert(0, "questions", per_question.groupby(column).size())
    display(table)

,questions,BM25,embeddings,fused (RRF),embeddings + rerank,fused + rerank
question_type,,,,,,
domain-relevant,40,0.100,0.525,0.300,0.575,0.475
metrics-generated,40,0.025,0.850,0.275,0.700,0.650
novel-generated,40,0.350,0.650,0.500,0.750,0.725


,questions,BM25,embeddings,fused (RRF),embeddings + rerank,fused + rerank
doc_type,,,,,,
10k,88,0.068,0.693,0.295,0.659,0.557
10q,14,0.357,0.643,0.500,0.643,0.643
8k,7,0.714,1.000,0.857,1.000,1.000
Earnings,11,0.273,0.364,0.364,0.636,0.818


## 6. Comparison with published baselines

Kobeissi and Langlais (2026) report results for the same kinds of methods on the 150 FinanceBench questions. Our methods are scored on all 150 questions with their two metrics, DocRec@5 and PageRec@5.

Their setup differs from ours in three ways:

- Their top 5 is the 5 best chunks. Several of them can come from the same page, so they may cover fewer than 5 pages. Our top 5 is always 5 different pages, which gives more chances to include an evidence page.
- Their chunks are 1024 tokens with 128 overlap. Ours are 250 words with 50 overlap.
- Their dense model is BGE-M3. Ours is Qwen3-Embedding-0.6B. Both use the same reranker.

In [12]:
# Kobeissi and Langlais (2026), Table 3: 150 FinanceBench questions, k = 5.
published = [
    ("BM25", 0.32, 0.07),
    ("Dense (BGE-M3)", 0.88, 0.34),
    ("Hybrid fusion (BM25 + BGE-M3)", 0.61, 0.23),
    ("BGE-M3 + reranker", 0.87, 0.41),
    ("BGE-M3 + Multi-HyDE + reranker (their best baseline)", 0.93, 0.46),
    ("Learned page scorer (their method)", 0.95, 0.55),
]

rows = [
    {"source": "Kobeissi and Langlais (2026)", "method": method, "DocRec@5": doc, "PageRec@5": page, "PageRec@5 95% CI": "not reported"}
    for method, doc, page in published
]
for name, ranking in rankings.items():
    rows.append({
        "source": "this project, all 150",
        "method": name,
        "DocRec@5": evaluate_retrieval(ranking, questions)["doc_recall@5"],
        "PageRec@5": round(page_fractions(ranking, questions).mean(), 3),
        "PageRec@5 95% CI": bootstrap_ci(page_fractions(ranking, questions)),
    })
pd.DataFrame(rows)

,source,method,DocRec@5,PageRec@5,PageRec@5 95% CI
0,Kobeissi and Langlais (2026),BM25,0.320,0.070,not reported
1,Kobeissi and Langlais (2026),Dense (BGE-M3),0.880,0.340,not reported
2,Kobeissi and Langlais (2026),Hybrid fusion (BM25 + BGE-M3),0.610,0.230,not reported
3,Kobeissi and Langlais (2026),BGE-M3 + reranker,0.870,0.410,not reported
4,Kobeissi and Langlais (2026),BGE-M3 + Multi-HyDE + reranker (their best bas...,0.930,0.460,not reported
5,Kobeissi and Langlais (2026),Learned page scorer (their method),0.950,0.550,not reported
6,"this project, all 150",BM25,0.647,0.153,"[0.100, 0.213]"
7,"this project, all 150",embeddings,0.967,0.586,"[0.509, 0.660]"
8,"this project, all 150",fused (RRF),0.913,0.317,"[0.247, 0.390]"
9,"this project, all 150",embeddings + rerank,0.980,0.587,"[0.513, 0.660]"


The methods rank in the same order in both, with BM25 lowest and fusion below dense search. Our PageRec@5 values are higher, which can partly come from our top 5 always covering 5 pages. Because of these differences, the published numbers are a reference point and not a direct comparison.

## 7. What the misses look like

Each test question without an evidence page in the top 5 gets the first category that fits.

- **adjacent page** means the page before or after an evidence page is in the top 5
- **right report, other page** means other pages of the right report are in the top 5
- **same company, other report** means only other reports of the same company are in the top 5, for example another year
- **other company** means no page of the right company is in the top 5

In [13]:
def miss_category(q, ranked):
    gold = gold_pages(q)
    top = ranked[:5]
    if any(page in gold for page in top):
        return "found"
    if any((doc, page_num + step) in gold for doc, page_num in top for step in (-1, 1)):
        return "adjacent page"
    if any(doc == q["doc_name"] for doc, _ in top):
        return "right report, other page"
    if any(doc_info[doc]["company"] == doc_info[q["doc_name"]]["company"] for doc, _ in top):
        return "same company, other report"
    return "other company"


order = ["found", "adjacent page", "right report, other page", "same company, other report", "other company"]
categories = pd.DataFrame({name: [miss_category(q, ranking[q["financebench_id"]]) for q in test] for name, ranking in rankings.items()})
categories.apply(lambda column: column.value_counts()).reindex(order).fillna(0).astype(int)

,BM25,embeddings,fused (RRF),embeddings + rerank,fused + rerank
found,19,81,43,81,74
adjacent page,4,8,19,6,12
"right report, other page",56,26,52,31,32
"same company, other report",11,4,4,1,2
other company,30,1,2,1,0


An adjacent page can hold the same evidence, since tables often continue onto the next page. The cell below lists these cases for fused + rerank, so they can be checked by hand.

In [14]:
pd.set_option("display.max_colwidth", None)
rows = []
for q in test:
    ranked = rankings[handoff][q["financebench_id"]]
    if miss_category(q, ranked) == "adjacent page":
        rows.append({
            "question": q["question"][:110],
            "report": q["doc_name"],
            "evidence pages": sorted(page_num for _, page_num in gold_pages(q)),
            "pages of this report in the top 5": [page_num for doc, page_num in ranked[:5] if doc == q["doc_name"]],
        })
pd.DataFrame(rows)

,question,report,evidence pages,pages of this report in the top 5
0,What is Amcor's year end FY2020 net AR (in USD millions)? Address the question by adopting the perspective of,AMCOR_2020_10K,[49],"[27, 47, 50, 80, 105]"
1,Did AMD report customer concentration in FY22?,AMD_2022_10K,[11],"[42, 3, 10, 23, 4]"
2,What was the largest liability in American Express's Balance Sheet in 2022?,AMERICANEXPRESS_2022_10K,[97],"[84, 63, 151, 98, 129]"
3,What is the FY2019 - FY2020 total revenue growth rate for Block (formerly known as Square)? Answer in units of,BLOCK_2020_10K,[85],"[61, 65, 62, 86]"
4,We need to calculate a financial metric by using information only provided within the balance sheet. Please an,BOEING_2018_10K,[51],"[84, 71, 52, 109]"
5,Is Boeing's business subject to cyclicality?,BOEING_2022_10K,[7],"[6, 11, 62]"
6,How does Boeing's effective tax rate in FY2022 compare to FY2021?,BOEING_2022_10K,[54],"[78, 76, 55, 75]"
7,What is Coca Cola's FY2022 dividend payout ratio (using total cash dividends paid and net income attributable,COCACOLA_2022_10K,"[62, 65]","[56, 66, 86, 85, 63]"
8,"Has CVS Health reported any materially important ongoing legal battles from 2022, 2021 and 2020?",CVSHEALTH_2022_10K,"[172, 173]","[19, 174, 18, 48, 132]"
9,Which of JPM's business segments had the lowest net revenue in 2021 Q1?,JPMORGAN_2021Q1_10Q,[18],"[27, 159, 55, 3, 17]"


## Summary of Results

- Embeddings and embeddings + rerank reach the highest recall@5 on the test questions, followed by fused + rerank, fused and BM25.
- The dev scores match notebooks 03 to 05, and fused + rerank is passed to the next phase of the project.
- The published results rank BM25, fusion and dense search in the same order.